In [1]:
import numpy as np
import pandas as pd
import os

train_df = pd.read_csv('/kaggle/input/competitions/optiver-trading-at-the-close/train.csv')
train_df_bytime = train_df.sort_values(['date_id', 'seconds_in_bucket'])
train_df_byid = train_df.sort_values(['stock_id', 'date_id', 'seconds_in_bucket'])

In [2]:
train_df.head()

In [3]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

features = ['stock_id', 'seconds_in_bucket', 'reference_price', 'matched_size',
       'far_price', 'near_price', 'wap', 'diff_wap',
       'diff_wap_60', 'diff_wap_300', 'diff_ref_wap', 'diff_ref_wap_60',
       'diff_ref_wap_300', 'spread_price_book', 'book_depth', 'imbalance_book_depth',
       'imbalance_book_flag_ratio', 'auction_depth', 'imbalance_auction_depth',
       'imbalance_auction_flag_ratio', 'spread_price_book_60',
       'spread_price_book_300', 'imbalance_book_depth_60', 'imbalance_book_depth_300',
       'imbalance_book_flag_ratio_60', 'imbalance_book_flag_ratio_300','imbalance_size_60',
       'imbalance_size_300',
       'imbalance_auction_flag_ratio_60', 'imbalance_auction_flag_ratio_300',
       'diff_book_ratio', 'diff_book_ratio_60', 'diff_auction_ratio',
       'diff_auction_ratio_60', 'diff_spread', 'diff_spread_60', 
       'diff_book_depth', 'diff_book_depth_60', 'diff_auction_depth',
       'diff_auction_depth_60', 'mid_price', 'imbalance_market_book_depth_ratio', 
        'imbalance_market_book_depth_ratio_60', 'imbalance_market_auction_depth_ratio',
        'imbalance_market_auction_depth_ratio_60', 'market_book_depth_ratio',
        'market_book_depth_ratio_60', 'market_auction_depth_ratio',
        'market_auction_depth_ratio_60', 'market_book_flag_ratio', 'market_book_flag_ratio_60',
        'excess_flag_ratio', 'excess_flag_ratio_60', 'wap_rel_market', 'wap_rel_market_60']

z_features = [
    'imbalance_book_flag_ratio_z', 'diff_ref_wap_z', 'spread_price_book_z',
    'diff_auction_ratio_z', 'imbalance_auction_flag_ratio_z', 'excess_flag_ratio_z',
    'imbalance_book_depth_z', 'imbalance_market_book_depth_ratio_z', 'wap_rel_market_z',
    'matched_size_z', 'book_depth_z', 'imbalance_size_z',
]

new_auction_and_diff_features = [
    'diff_near_wap', 'diff_far_wap', 'diff_near_far', 'diff_near_ref',
    'auction_book_depth_ratio',
    'diff_wap_rel_market', 'diff_wap_rel_market_60',
    'diff_excess_flag_ratio', 'diff_excess_flag_ratio_60',
    'diff_ref_wap_10', 'diff_ref_wap_change_60',
    'diff_near_price', 'diff_near_price_60',
    'diff_near_wap_10', 'diff_near_wap_60'
]

extra_features = [
    
    'diff_liquidity_weights', 'diff_liquidity_weights_60', 'ratio_liquidity_weights',
    'market_auction_flag_ratio', 'excess_auction_flag_ratio',
    'wap_rel_market_book',
    'price_pressure', 'ask_ref_spread', 'ref_bid_spread', 'ref_spread_asymmetry'
]

features = features + z_features + new_auction_and_diff_features + extra_features

target = 'target'

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'device': 'gpu',
    
    'learning_rate': 0.015,        
    'n_estimators': 2500,          
    
    'num_leaves': 63,             
    'max_depth': -1,             
    'min_child_samples': 300, 
    
    'subsample': 0.7,              
    'colsample_bytree': 0.5,       
    'subsample_freq': 1,
    
    'reg_alpha': 0.1,              
    'reg_lambda': 1.0,
    
    'random_state': 42,
    'n_jobs': -1
}

In [4]:
"""split_date = int(train_df_byid["date_id"].max())*0.8

train_mask = train_df_byid["date_id"] <= split_date
valid_mask = train_df_byid["date_id"] > split_date

train_set = train_df_byid[train_mask]
valid_set = train_df_byid[valid_mask]

train_clean = train_set.dropna(subset=[target])
valid_clean = valid_set.dropna(subset=[target])

X_train, y_train = train_clean[features], train_clean[target]
X_val, y_val = valid_clean[features], valid_clean[target]

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)

y_pred = model.predict(X_val)
#we center the predictions to preserve the zero-sum property of the target (which is tied to no-arbitrage conditions)
y_pred = y_pred - pd.Series(y_pred * valid_clean['market_auction_depth_ratio'], index=X_val.index).groupby([valid_clean['date_id'], valid_clean['seconds_in_bucket']]).transform('sum')
score = mean_absolute_error(y_val, y_pred)
print(f"Validation MAE : {score:.5f}")"""

In [5]:
"""import matplotlib.pyplot as plt

lgb.plot_importance(model, max_num_features=15, importance_type='gain')
plt.show()"""

In [6]:
import sys
import os

sys.path.append('/kaggle/input/competitions/optiver-trading-at-the-close/')

import optiver2023
import pandas as pd
import numpy as np

np


def apply_feature_engineering(df):

    grouper = df.groupby(['stock_id', 'date_id'])

    df['diff_wap'] = grouper['wap'].diff()
    df['diff_wap_60'] = grouper['wap'].diff(6)
    df['diff_wap_300'] = grouper['wap'].diff(30)
    
    mean_wap_60 = grouper['wap'].transform(lambda x : x.rolling(6).mean())
    mean_wap_300 = grouper['wap'].transform(lambda x : x.rolling(30).mean())
    mean_ref_60 = grouper['reference_price'].transform(lambda x : x.rolling(6).mean())
    mean_ref_300 = grouper['reference_price'].transform(lambda x : x.rolling(30).mean())
    
    df['diff_ref_wap'] = df['reference_price'] - df['wap']
    df['diff_ref_wap_60'] = mean_ref_60 - mean_wap_60
    df['diff_ref_wap_300'] = mean_ref_300 - mean_wap_300

    # Price book spread & depths
    df['spread_price_book'] = df['ask_price'] - df['bid_price']
    df['book_depth'] = df['ask_size'] + df['bid_size']
    df['imbalance_book_depth'] = df['bid_size'] - df['ask_size']
    df['imbalance_book_flag_ratio'] = (df['bid_size'] - df['ask_size']) / (df['ask_size'] + df['bid_size'] + 1e-8)
    
    df['auction_depth'] = df['imbalance_size'] + df['matched_size']
    df['imbalance_auction_depth'] = df['imbalance_size'] 
    df['imbalance_auction_flag_ratio'] = (df['imbalance_buy_sell_flag'] * df['imbalance_size']) / (df['imbalance_size'] + df['matched_size'] + 1e-8)

    df['diff_book_ratio'] = grouper['imbalance_book_flag_ratio'].diff(1)
    df['diff_book_ratio_60'] = grouper['imbalance_book_flag_ratio'].diff(6)
    df['diff_auction_ratio'] = grouper['imbalance_auction_flag_ratio'].diff(1)
    df['diff_auction_ratio_60'] = grouper['imbalance_auction_flag_ratio'].diff(6)
    df['diff_spread'] = grouper['spread_price_book'].diff(1)
    df['diff_spread_60'] = grouper['spread_price_book'].diff(6)
    df['diff_book_depth'] = grouper['book_depth'].diff(1)
    df['diff_book_depth_60'] = grouper['book_depth'].diff(6)
    df['diff_auction_depth'] = grouper['imbalance_auction_depth'].diff(1)
    df['diff_auction_depth_60'] = grouper['imbalance_auction_depth'].diff(6)

    # Rolling price & size
    df['ask_price_60'] = grouper['ask_price'].transform(lambda x: x.rolling(6).mean())
    df['ask_price_300'] = grouper['ask_price'].transform(lambda x: x.rolling(30).mean())
    df['bid_price_60'] = grouper['bid_price'].transform(lambda x: x.rolling(6).mean())
    df['bid_price_300'] = grouper['bid_price'].transform(lambda x: x.rolling(30).mean())
    df['spread_price_book_60'] = df['ask_price_60'] - df['bid_price_60']
    df['spread_price_book_300'] = df['ask_price_300'] - df['bid_price_300']

    df['ask_size_60'] = grouper['ask_size'].transform(lambda x: x.rolling(6).mean())
    df['ask_size_300'] = grouper['ask_size'].transform(lambda x: x.rolling(30).mean())
    df['bid_size_60'] = grouper['bid_size'].transform(lambda x: x.rolling(6).mean())
    df['bid_size_300'] = grouper['bid_size'].transform(lambda x: x.rolling(30).mean())
    df['imbalance_size_60'] = grouper['imbalance_size'].transform(lambda x: x.rolling(6).mean())

    df['book_depth_60'] = df['ask_size_60'] + df['bid_size_60']
    df['book_depth_300'] = df['ask_size_300'] + df['bid_size_300']
    df['imbalance_book_depth_60'] = df['bid_size_60'] - df['ask_size_60']
    df['imbalance_book_depth_300'] = df['bid_size_300'] - df['ask_size_300']

    df['imbalance_book_flag_ratio_60'] = (df['bid_size_60'] - df['ask_size_60']) / (df['ask_size_60'] + df['bid_size_60'] + 1e-8)
    df['imbalance_book_flag_ratio_300'] = (df['bid_size_300'] - df['ask_size_300']) / (df['ask_size_300'] + df['bid_size_300'] + 1e-8)
    
    df['imbalance_size_300'] = grouper['imbalance_size'].transform(lambda x: x.rolling(30).mean())
    df['matched_size_60'] = grouper['matched_size'].transform(lambda x: x.rolling(6).mean())
    df['matched_size_300'] = grouper['matched_size'].transform(lambda x: x.rolling(30).mean())

    df['auction_depth_60'] = df['imbalance_size_60'] + df['matched_size_60']
    df['auction_depth_300'] = df['imbalance_size_300'] + df['matched_size_300']
    
    df['signed_imbalance_size'] = df['imbalance_buy_sell_flag'] * df['imbalance_size']
    df['signed_imbalance_size_60'] = grouper['signed_imbalance_size'].transform(lambda x: x.rolling(6).mean())
    df['signed_imbalance_size_300'] = grouper['signed_imbalance_size'].transform(lambda x: x.rolling(30).mean())

    df['imbalance_auction_flag_ratio_60'] = df['signed_imbalance_size_60'] / (df['imbalance_size_60'] + df['matched_size_60'] + 1e-8)
    df['imbalance_auction_flag_ratio_300'] = df['signed_imbalance_size_300'] / (df['imbalance_size_300'] + df['matched_size_300'] + 1e-8)
    
    df['mid_price'] = (df['ask_price'] + df['bid_price']) / 2

    # Price pressures & spreads
    df['price_pressure'] = df['wap'] - df['mid_price']
    df['ask_ref_spread'] = df['ask_price'] - df['reference_price']
    df['ref_bid_spread'] = df['reference_price'] - df['bid_price']
    df['ref_spread_asymmetry'] = df['ask_ref_spread'] / (df['ref_bid_spread'] + 1e-8)

    # Market information
    market_imbalance_total_book_depth = df['imbalance_book_depth'].abs().groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['imbalance_market_book_depth_ratio'] = df['imbalance_book_depth'] / (market_imbalance_total_book_depth + 1e-8)
    
    market_imbalance_total_book_depth_60 = df['imbalance_book_depth_60'].abs().groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['imbalance_market_book_depth_ratio_60'] = df['imbalance_book_depth_60'] / (market_imbalance_total_book_depth_60 + 1e-8)
    
    market_imbalance_total_auction_depth = df.groupby(['date_id', 'seconds_in_bucket'])['imbalance_auction_depth'].transform('sum')
    df['imbalance_market_auction_depth_ratio'] = df['imbalance_auction_depth'] / (market_imbalance_total_auction_depth + 1e-8)
    
    market_imbalance_total_auction_depth_60 = df.groupby(['date_id', 'seconds_in_bucket'])['imbalance_size_60'].transform('sum')
    df['imbalance_market_auction_depth_ratio_60'] = df['imbalance_size_60'] / (market_imbalance_total_auction_depth_60 + 1e-8)
    
    market_total_book_depth = df.groupby(['date_id', 'seconds_in_bucket'])['book_depth'].transform('sum')
    df['market_book_depth_ratio'] = df['book_depth'] / (market_total_book_depth + 1e-8)
    
    market_total_book_depth_60 = df.groupby(['date_id', 'seconds_in_bucket'])['book_depth_60'].transform('sum')
    df['market_book_depth_ratio_60'] = df['book_depth_60'] / (market_total_book_depth_60 + 1e-8)
    
    market_total_auction_depth = df.groupby(['date_id', 'seconds_in_bucket'])['matched_size'].transform('sum') + df.groupby(['date_id', 'seconds_in_bucket'])['imbalance_size'].transform('sum')
    df['market_auction_depth_ratio'] = df['auction_depth'] / (market_total_auction_depth + 1e-8)
    
    market_total_auction_depth_60 = df.groupby(['date_id', 'seconds_in_bucket'])['auction_depth_60'].transform('sum')
    df['market_auction_depth_ratio_60'] = df['auction_depth_60'] / (market_total_auction_depth_60 + 1e-8)

    # Weights diffs & ratios
    df['diff_liquidity_weights'] = df['market_book_depth_ratio'] - df['market_auction_depth_ratio']
    df['diff_liquidity_weights_60'] = df['market_book_depth_ratio_60'] - df['market_auction_depth_ratio_60']
    df['ratio_liquidity_weights'] = df['market_book_depth_ratio'] / (df['market_auction_depth_ratio'] + 1e-8)

    # Market flags
    market_bid = df.groupby(['date_id', 'seconds_in_bucket'])['bid_size'].transform('sum')
    market_ask = df.groupby(['date_id', 'seconds_in_bucket'])['ask_size'].transform('sum')
    df['market_book_flag_ratio'] = (market_bid - market_ask) / (market_bid + market_ask + 1e-8)
    
    market_bid_60 = df.groupby(['date_id', 'seconds_in_bucket'])['bid_size_60'].transform('sum')
    market_ask_60 = df.groupby(['date_id', 'seconds_in_bucket'])['ask_size_60'].transform('sum')
    df['market_book_flag_ratio_60'] = (market_bid_60 - market_ask_60) / (market_bid_60 + market_ask_60 + 1e-8)
    
    df['excess_flag_ratio'] = df['imbalance_book_flag_ratio'] - df['market_book_flag_ratio']
    df['excess_flag_ratio_60'] = df['imbalance_book_flag_ratio_60'] - df['market_book_flag_ratio_60']
    
    market_signed_imb = df.groupby(['date_id', 'seconds_in_bucket'])['signed_imbalance_size'].transform('sum')
    df['market_auction_flag_ratio'] = market_signed_imb / (market_total_auction_depth + 1e-8)
    df['excess_auction_flag_ratio'] = df['imbalance_auction_flag_ratio'] - df['market_auction_flag_ratio']
    
    market_signed_imb_60 = df.groupby(['date_id', 'seconds_in_bucket'])['signed_imbalance_size_60'].transform('sum')
    df['market_auction_flag_ratio_60'] = market_signed_imb_60 / (market_total_auction_depth_60 + 1e-8)
    df['excess_auction_flag_ratio_60'] = df['imbalance_auction_flag_ratio_60'] - df['market_auction_flag_ratio_60']

    # Market WAPs
    wap_x_auction_w = df['wap'] * df['market_auction_depth_ratio']
    df['market_wap_auction_w'] = wap_x_auction_w.groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['wap_rel_market'] = df['wap'] - df['market_wap_auction_w']
    
    wap60_x_auction_w60 = mean_wap_60 * df['market_auction_depth_ratio_60']
    df['market_wap_auction_w_60'] = wap60_x_auction_w60.groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['wap_rel_market_60'] = mean_wap_60 - df['market_wap_auction_w_60']
    
    wap_x_book_w = df['wap'] * df['market_book_depth_ratio']
    df['market_wap_book_w'] = wap_x_book_w.groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['wap_rel_market_book'] = df['wap'] - df['market_wap_book_w']
    
    wap60_x_book_w60 = mean_wap_60 * df['market_book_depth_ratio_60']
    df['market_wap_book_w_60'] = wap60_x_book_w60.groupby([df['date_id'], df['seconds_in_bucket']]).transform('sum')
    df['wap_rel_market_book_60'] = mean_wap_60 - df['market_wap_book_w_60']

    # Causal Z-scores
    def add_causal_z(dframe, cols, window=20, min_days=3):
        daily = dframe.groupby(['stock_id', 'date_id'])[cols].agg(['mean', 'std']).sort_index()
        rolling_stats = (
            daily.groupby(level=0)
            .rolling(window=window, min_periods=min_days)
            .mean()
            .droplevel(0)
            .groupby(level=0)
            .shift(1)
        )
        idx = pd.MultiIndex.from_frame(dframe[['stock_id', 'date_id']])
        for c in cols:
            m = rolling_stats[(c, 'mean')].reindex(idx).to_numpy(dtype='float32')
            s = rolling_stats[(c, 'std')].reindex(idx).to_numpy(dtype='float32')
            dframe[f'{c}_z'] = ((dframe[c].to_numpy(dtype='float32') - m) / (s + 1e-8))
        return dframe

    colums_z_add = [
        'imbalance_book_flag_ratio', 'diff_ref_wap', 'spread_price_book',
        'diff_auction_ratio', 'imbalance_auction_flag_ratio', 'excess_flag_ratio',
        'imbalance_book_depth', 'imbalance_market_book_depth_ratio', 'wap_rel_market',
        'matched_size', 'book_depth', 'imbalance_size'
    ]
    df = add_causal_z(df, colums_z_add)

    # Auction price spreads & diffs
    df['diff_near_wap'] = df['near_price'] - df['wap']
    df['diff_far_wap'] = df['far_price'] - df['wap']
    df['diff_near_far'] = df['near_price'] - df['far_price']
    df['diff_near_ref'] = df['near_price'] - df['reference_price']
    df['auction_book_depth_ratio'] = df['auction_depth'] / (df['auction_depth'] + df['book_depth'] + 1e-8)
    
    g = df.groupby(['stock_id', 'date_id'])
    df['diff_wap_rel_market'] = g['wap_rel_market'].diff(1)
    df['diff_wap_rel_market_60'] = g['wap_rel_market'].diff(6)
    df['diff_excess_flag_ratio'] = g['excess_flag_ratio'].diff(1)
    df['diff_excess_flag_ratio_60'] = g['excess_flag_ratio'].diff(6)
    df['diff_ref_wap_10'] = g['diff_ref_wap'].diff(1)
    df['diff_ref_wap_change_60'] = g['diff_ref_wap'].diff(6)
    df['diff_near_price'] = g['near_price'].diff(1)
    df['diff_near_price_60'] = g['near_price'].diff(6)
    df['diff_near_wap_10'] = g['diff_near_wap'].diff(1)
    df['diff_near_wap_60'] = g['diff_near_wap'].diff(6)

    return df

train_df_byid = apply_feature_engineering(train_df_byid)

print("Train on 100% data...")
train_clean = train_df_byid.dropna(subset=['target'])
X_all, y_all = train_clean[features], train_clean['target']

final_model = lgb.LGBMRegressor(**params)
final_model.fit(X_all, y_all)

# Kaggle API & History Buffer
env = optiver2023.make_env()
iter_test = env.iter_test()

#keep in memory for rolling
history_df = train_df_byid.copy()

print("Real-time predict...")
for (test, revealed_targets, sample_prediction) in iter_test:
    
    # new chunk
    history_df = pd.concat([history_df, test], ignore_index=True)
    
    # feature engineering
    processed_history = apply_feature_engineering(history_df)
    
    # past chunk 
    test_processed = processed_history.tail(len(test)).copy()
    
    # Predict
    preds = final_model.predict(test_processed[features])
    
    # we center the predictions to preserve the zero-sum property of the target (which is tied to no-arbitrage conditions)
    w_auction = test_processed['market_auction_depth_ratio'].to_numpy()
    shift = pd.Series(preds * w_auction, index=test_processed.index).groupby(
        [test_processed['date_id'], test_processed['seconds_in_bucket']]
    ).transform('sum').to_numpy()
    
    preds_neutral = preds - shift
    
    # Send
    sample_prediction['target'] = preds_neutral
    env.predict(sample_prediction)

print("Submission completed successfully!")